# Landmark Grid-Search Results Analysis

Inspect the pickled outputs of `landmarks/scripts/grid_search.py` (`grid_search_results_temporal_features*.pkl`). Tabulates F1 across the four splits (balanced / LOSO / signant / signant-by-sign), aggregates Random Forest feature importances per landmark group, and renders side-by-side IMU/landmark importance comparisons for the signant task.

## Configuration

In [ ]:
FEATURES_FILE = "features2.txt"
NEW_SCHEMA_PKL = "grid_search_results_temporal_features_new_schema.pkl"
STANDARD_PKL = "grid_search_results_temporal_features.pkl"
IMU_IMPORTANCE_PKL = "feature_importance_data.pkl"   # produced by imu/notebooks/results_analysis.ipynb


## Setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
from tabulate import tabulate
import glob
import matplotlib.pyplot as plt
import matplotlib

matplotlib.style.use('seaborn-v0_8-darkgrid')

## Load feature names

In [ ]:
with open(FEATURES_FILE,"r") as f:
    features = f.readlines()
features = [f.strip() for f in features]
features = list(filter(lambda x: len(x) > 0, features))
features_list = list(filter(lambda x: x not in ["Unnamed: 0", "label", "subject","rep"], features))

## Load new-schema results

In [ ]:

database = NEW_SCHEMA_PKL
feature_importances_dict = {}
agg_res_importance = {}
name = " ".join(database.split('.')[0].split("_")[3:])
print(f"Results for {name}")
print("="*50)

handle = pickle.load(open(database, 'rb'))

signant_results = handle["signant_part"]

signant_agg = {}
for k,v in signant_results.items():
    f1_score = v["f1_score"]
    model_name,db = k.split("-")
    signant_agg[model_name] = f1_score
    if "feature_importances" in v:
        print(len(v["feature_importances"]))
        feature_importances_dict["signant_part"] = dict(zip(features_list, v["feature_importances"]))

print("SIGNANT-NONSIGNANT")
print(tabulate(signant_agg.items(),["Model","F1 Score"]))
print("--"*30)


for key, value in feature_importances_dict["signant_part"].items():
    landmark = key.split("_")
    landmark_name = "_".join([landmark[0], landmark[1], landmark[3]])
    agg_res_importance["signant_part"] = agg_res_importance.get("signant_part", {})
    agg_res_importance["signant_part"][landmark_name] = agg_res_importance["signant_part"].get(landmark_name, 0) + value


#I need to sort the agg_res_importance
for part in agg_res_importance:
    agg_res_importance[part] = dict(sorted(agg_res_importance[part].items(), key=lambda item: item[1], reverse=True))

limit = 4


leave_signs_out_importance = {}

for key in agg_res_importance.keys():
    leave_signs_out_importance[key] = {}
    name = key.split("_")[0]
    if name.lower() == "random":
        name = "80-20 split"
    x_names = list(agg_res_importance[key].keys())[:limit]
    x_names = list(map(lambda x: " ".join(list(map(lambda x: x.capitalize(), x.split("_")))), x_names))
    x_vals = list(agg_res_importance[key].values())[:limit]
    leave_signs_out_importance[key]["x_names"] = x_names
    leave_signs_out_importance[key]["x_vals"] = x_vals
    plt.figure(figsize=(8,5))
    plt.bar(x_names, x_vals)
    plt.title(f"Feature Importances - Leave 10 signs out")
    plt.xlabel("Features")
    plt.ylabel("Importance")
    plt.xticks(rotation=45)
    plt.setp(plt.gca().get_xticklabels(), ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
feature_importances_dict

In [ ]:
len(features_list)

## Load standard results and aggregate importances

In [ ]:
feature_importances_dict = {}

database = NEW_SCHEMA_PKL

name = " ".join(database.split('.')[0].split("_")[3:])
print(f"Results for {name}")
print("="*50)

handle = pickle.load(open(database, 'rb'))

signant_results = handle["signant_part"]

signant_agg = {}
for k,v in signant_results.items():
    f1_score = v["f1_score"]
    model_name,db = k.split("-")
    signant_agg[model_name] = f1_score
    if "feature_importances" in v:
        print(len(v["feature_importances"]))
        feature_importances_dict["signant_part"] = dict(zip(features_list, v["feature_importances"]))

print("SIGNANT-NONSIGNANT")
print(tabulate(signant_agg.items(),["Model","F1 Score"]))
print("--"*30)

agg_res_importance = {}

for key, value in feature_importances_dict["signant_part"].items():
    landmark = key.split("_")
    landmark_name = "_".join([landmark[0], landmark[1], landmark[3]])
    agg_res_importance["signant_part"] = agg_res_importance.get("signant_part", {})
    agg_res_importance["signant_part"][landmark_name] = agg_res_importance["signant_part"].get(landmark_name, 0) + value


#I need to sort the agg_res_importance
for part in agg_res_importance:
    agg_res_importance[part] = dict(sorted(agg_res_importance[part].items(), key=lambda item: item[1], reverse=True))


limit = 4
matplotlib.style.use('seaborn-v0_8-darkgrid')



In [ ]:
feature_importances_dict = {}

database = STANDARD_PKL

name = " ".join(database.split('.')[0].split("_")[3:])
print(f"Results for {name}")
print("="*50)

handle = pickle.load(open(database, 'rb'))

balanced_results = handle["balanced_part"]
loso_results = handle["loso_part"]
signant_results = handle["signant_part"]



balanced_agg = {}
for k,v in balanced_results.items():
    f1_score = v["f1_score"]
    model_name,database = k.split("-")
    database = database.split(".")[0]
    balanced_agg[model_name] = f1_score
    if "feature_importances" in v:
        print(len(v["feature_importances"]))
        feature_importances_dict["random_part"] = dict(zip(features_list, v["feature_importances"]))

print("RANDOM PARTITION\n")

print(tabulate(balanced_agg.items(),["Model","F1 Score"]))
print("--"*30)

loso_results_agg = {}

for k,v in loso_results.items():
    f1_score = v["f1_score"]
    model_name,subject = k.split("-")
    loso_results_agg[model_name] = loso_results_agg.get(model_name, [])
    loso_results_agg[model_name].append(f1_score)
    if "feature_importances" in v:
        print(len(v["feature_importances"]))
        if "loso_part" not in feature_importances_dict:
            feature_importances_dict["loso_part"] = []
        feature_importances_dict["loso_part"].append(dict(zip(features_list, v["feature_importances"])))

for model_name, scores in loso_results_agg.items():
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    loso_results_agg[model_name] = f"{mean_score:.2f} +- {std_score:.2f}"

print("LOSO RESULTS")
print(tabulate(loso_results_agg.items(),["Model","F1 Score"]))
print("--"*30)
signant_agg = {}
for k,v in signant_results.items():
    f1_score = v["f1_score"]
    model_name,db = k.split("-")
    signant_agg[model_name] = f1_score
    if "feature_importances" in v:
        print(len(v["feature_importances"]))
        feature_importances_dict["signant_part"] = dict(zip(features_list, v["feature_importances"]))

print("SIGNANT-NONSIGNANT")
print(tabulate(signant_agg.items(),["Model","F1 Score"]))
print("--"*30)



In [ ]:
feature_importances_dict

## Aggregate per landmark group

In [ ]:
agg_res_importance = {}

for key, value in feature_importances_dict["random_part"].items():
    landmark = key.split("_")
    landmark_name = "_".join([landmark[0], landmark[1], landmark[3]])
    agg_res_importance["random_part"] = agg_res_importance.get("random_part", {})
    agg_res_importance["random_part"][landmark_name] = agg_res_importance["random_part"].get(landmark_name, 0) + value

for value in feature_importances_dict["loso_part"]:
    for sub_key, sub_value in value.items():
        print(sub_key, sub_value)
        landmark = sub_key.split("_")
        landmark_name = "_".join([landmark[0], landmark[1], landmark[3]])
        agg_res_importance["loso_part"] = agg_res_importance.get("loso_part", {})
        agg_res_importance["loso_part"][landmark_name] = agg_res_importance["loso_part"].get(landmark_name, [])
        agg_res_importance["loso_part"][landmark_name].append(sub_value)

for key, value in feature_importances_dict["signant_part"].items():
    landmark = key.split("_")
    landmark_name = "_".join([landmark[0], landmark[1], landmark[3]])
    agg_res_importance["signant_part"] = agg_res_importance.get("signant_part", {})
    agg_res_importance["signant_part"][landmark_name] = agg_res_importance["signant_part"].get(landmark_name, 0) + value

for key in agg_res_importance["loso_part"]:
    agg_res_importance["loso_part"][key] = np.sum(agg_res_importance["loso_part"][key])/5


#I need to sort the agg_res_importance
for part in agg_res_importance:
    agg_res_importance[part] = dict(sorted(agg_res_importance[part].items(), key=lambda item: item[1], reverse=True))


## Plot importance bars

In [ ]:
for task in agg_res_importance.keys():
    names = list(agg_res_importance[task].keys())
    values = list(agg_res_importance[task].values())
    plt.figure(figsize=(10,6))
    plt.barh(names, values)
    plt.xlabel("Feature Importance")
    plt.title(f"Feature Importance for {task} task - {sum(values)}")
    plt.gca().invert_yaxis()
    plt.show()

## Side-by-side IMU vs landmark plot

In [ ]:
import matplotlib

In [ ]:
"""for key in agg_res_importance.keys():
    plt.figure(figsize=(10, 6))
    plt.bar(list(agg_res_importance[key].keys())[:5], list(agg_res_importance[key].values())[:5])
    plt.title(f"Feature Importances - {key}")
    plt.xlabel("Features")
    plt.ylabel("Importance")
    plt.xticks(rotation=45)
    plt.setp(plt.gca().get_xticklabels(), ha='right')
    plt.tight_layout()
    plt.show()
"""
limit = 4
matplotlib.style.use('seaborn-v0_8-darkgrid')
import os

imu_data = None
if os.path.exists("feature_importance_data copy.pkl") and os.path.getsize("feature_importance_data copy.pkl") > 0:
    with open("feature_importance_data copy.pkl", "rb") as f:
        imu_data = pickle.load(f)
else:
    print("feature_importance_data copy.pkl does not exist or is empty.")

landmark_data = {}

for key in agg_res_importance.keys():
    name_key = (key if key != "random_part" else "balanced_part")
    landmark_data[name_key] = {}
    name = key.split("_")[0]
    if name.lower() == "random":
        name = "80-20 split"
    x_names = list(agg_res_importance[key].keys())[:limit]
    x_names = list(map(lambda x: " ".join(list(map(lambda x: x.capitalize(), x.split("_")))), x_names))
    x_vals = list(agg_res_importance[key].values())[:limit]
    landmark_data[name_key]["x_names"] = x_names
    landmark_data[name_key]["x_vals"] = x_vals
    plt.figure(figsize=(8,5))
    plt.bar(x_names, x_vals)
    plt.title(f"Feature Importances - {name.upper()}")
    plt.xlabel("Features")
    plt.ylabel("Importance")
    plt.xticks(rotation=45)
    plt.setp(plt.gca().get_xticklabels(), ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
signant_data = {"imu": imu_data["signant_part"], "landmark": landmark_data["signant_part"]}

imu_data = {k:v for k,v in imu_data.items() if k != "signant_part"}
landmark_data = {k:v for k,v in landmark_data.items() if k != "signant_part"}

In [ ]:
import matplotlib.font_manager as fm
fonts = sorted([f.name for f in fm.fontManager.ttflist])
print(fonts)

In [ ]:
import matplotlib

# set Helvetica as the default font (use sans-serif alias pointing to Helvetica)
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
# ensure vector backends embed fonts properly
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

correspondance_dict = {
    'Landmark 13 B':'Right elbow',
    'Landmark 16 B':'Left Wrist',
    'Landmark 15 B':'Right Wrist',
    'Landmark 14 B':'Left Elbow',
    'Landmark 20 B':'Left index knuckle #1',
    'Landmark 4 Rh':'Right hand thumb tip',
    'Landmark 12 B':'Left Shoulder',
    'Landmark 8 B':'Left Ear',
    'Landmark 11 B':'Right Shoulder',
    'Landmark 5 B': 'Left eye',
    'Landmark 3 B':'Right Eye Outer',
    'Arm Flex R':'Right Arm Flexion',
    'Arm Flex L':'Left Arm Flexion',
    'Arm Rot R':'Right Arm Rotation',
    'Arm Rot L':'Left Arm Rotation',
    'Wrist Flex L':'Left Wrist Flexion',
    'Pro Sup L':'Left Arm Prono-supination',
    'Elbow Flex L':'Left Elbow Flexion',
    'Elbow Flex R':'Right Elbow Flexion',
}


fig, ax = plt.subplots(2,2, figsize=(9,9))
dict_name_fig = {"balanced_part": "80-20 split", "loso_part": "Leave One Subject Out"}

var = ['(a)', '(b)', '(c)', '(d)']
it = 0

for indx, key in enumerate(imu_data.keys()):
    imu_info = imu_data[key]
    imu_info["x_names"] = [correspondance_dict.get(name, name) for name in imu_info["x_names"]]
    landmark_info = landmark_data.get(key, None)
    landmark_info["x_names"] = [correspondance_dict.get(name, name) for name in landmark_info["x_names"]]

    name = dict_name_fig.get(key, key.replace('_',' ')).replace("Leave One Subject Out","LOSO")

    ax[indx,0].bar(imu_info["x_names"], imu_info["x_vals"], color='skyblue')
    ax[indx,0].set_ylabel("Importance", fontsize=16)
    ax[indx,0].set_xlabel(f"{var[it]} {name} (IMU)", fontsize=16)
    ax[indx,0].set_xticklabels(imu_info["x_names"], rotation=45, ha='right', fontsize=14)
    ax[indx,0].tick_params(axis='y', labelsize=14)

    it += 1

    ax[indx,1].bar(landmark_info["x_names"], landmark_info["x_vals"], color='salmon')
    ax[indx,1].set_ylabel("Importance", fontsize=16)
    ax[indx,1].set_xlabel(f"{var[it]} {name} (Landmark)", fontsize=16)
    ax[indx,1].set_xticklabels(landmark_info["x_names"], rotation=45, ha='right', fontsize=14)
    ax[indx,1].tick_params(axis='y', labelsize=14)

    it += 1

fig.set_tight_layout(True)
fig.suptitle("Feature Importances (SLR)", fontsize=20)
plt.savefig("feature_importances_slr.png", format='png',dpi = 300)
plt.show()


In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(11,11))
gs = fig.add_gridspec(2, 2)

# Main bar plot (top, spanning two columns)
signant_data["imu"]["x_names"] = [correspondance_dict.get(name, name) for name in signant_data["imu"]["x_names"]]
ax_main = fig.add_subplot(gs[0, 0])
ax_main.bar(signant_data["imu"]["x_names"], signant_data["imu"]["x_vals"], color='skyblue')
ax_main.set_xlabel("(a) 80-20 Split (IMU)", fontsize=20)
ax_main.set_ylabel("Importance", fontsize=18)
ax_main.set_xticklabels(signant_data["imu"]["x_names"], rotation=45, ha='right', fontsize=16)
ax_main.tick_params(axis='y', labelsize=16)

# Secondary plot (top right)
signant_data["landmark"]["x_names"] = [correspondance_dict.get(name, name) for name in signant_data["landmark"]["x_names"]]
ax_right = fig.add_subplot(gs[0, 1])
ax_right.bar(signant_data["landmark"]["x_names"], signant_data["landmark"]["x_vals"], color='salmon')
ax_right.set_xlabel("(b) 80-20 Split (Landmark)", fontsize=20)
ax_right.set_ylabel("Importance", fontsize=18)
ax_right.set_xticklabels(signant_data["landmark"]["x_names"], rotation=45, ha='right', fontsize=16)
ax_right.tick_params(axis='y', labelsize=16)

# Bottom plot (spanning all columns)
leave_signs_out_importance["signant_part"]["x_names"] = [correspondance_dict.get(name, name) for name in leave_signs_out_importance["signant_part"]["x_names"]]
ax_bottom = fig.add_subplot(gs[1, :])
ax_bottom.bar(leave_signs_out_importance["signant_part"]["x_names"], leave_signs_out_importance["signant_part"]["x_vals"], color='lightgreen')
ax_bottom.set_xlabel("(c) Leave 10 signs out", fontsize=20)
ax_bottom.set_ylabel("Importance", fontsize=18)
ax_bottom.set_xticks(range(len(leave_signs_out_importance["signant_part"]["x_names"])))
ax_bottom.set_xticklabels(leave_signs_out_importance["signant_part"]["x_names"], rotation=45, ha='right', fontsize=16)
ax_bottom.tick_params(axis='y', labelsize=16)
ax_bottom.tick_params(axis='x', labelsize=16)

plt.suptitle("Feature importance (native vs non-native)", fontsize=22)
plt.savefig("feature_importances_native_vs_non_native.png", format='png',dpi = 300)
plt.show()